# Flow Matching

Chapter 20: ODE, SDE, and Continuous Limits of Algorithms · *Mathematics for AI and Machine Learning* by Fuheng Wu

Straight training paths, curved non-crossing flow: the exact flow matching velocity carries noise to a 3-mode target.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-20/flow-matching/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
import torch.nn as nn

torch.manual_seed(0)
modes = torch.tensor([[2.5 * math.cos(math.pi / 2 + 2 * math.pi * k / 3),
                       2.5 * math.sin(math.pi / 2 + 2 * math.pi * k / 3)] for k in range(3)])


def sample_data(n):
    # Target p_1: three Gaussian modes on a circle
    return modes[torch.randint(0, 3, (n,))] + 0.25 * torch.randn(n, 2)


# v_theta(x, t): a small MLP on (x, t)
v = nn.Sequential(nn.Linear(3, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 2))
opt = torch.optim.Adam(v.parameters(), lr=3e-3)

# Flow matching objective: regress v_theta(x_t, t) on u = x1 - x0
# along the straight path x_t = (1 - t) x0 + t x1
for step in range(1500):
    x0, x1 = torch.randn(256, 2), sample_data(256)
    t = torch.rand(256, 1)
    xt = (1 - t) * x0 + t * x1
    loss = ((v(torch.cat([xt, t], dim=1)) - (x1 - x0)) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
print("final flow matching loss:", round(loss.item(), 3))

# Sample: integrate dx/dt = v_theta(x, t) from noise with Euler steps
with torch.no_grad():
    x = torch.randn(3000, 2)
    for i in range(50):
        t = torch.full((3000, 1), i / 50)
        x = x + v(torch.cat([x, t], dim=1)) / 50
nearest = torch.cdist(x, modes).argmin(dim=1)
print("share of samples per mode:", [round((nearest == k).float().mean().item(), 3) for k in range(3)])

final flow matching loss: 1.809
share of samples per mode: [0.336, 0.332, 0.332]
